# Exportar contornos para a página do Boletim do Monitor da Água

Rodar **uma vez** (ou quando os limites mudarem): `Ambiente de execução → Executar tudo`.
Gera `geo/bioma.geojson`, `geo/estado.geojson`, `geo/hidro.geojson` e `geo/subbacia1.geojson` no Drive. Copie a pasta `geo/` para o repositório da página.

In [ ]:
# ==========================================================
# EXPORTAR CONTORNOS PARA A PÁGINA DO BOLETIM (rodar 1 vez, ou quando os limites mudarem)
# Gera geo/bioma.geojson, geo/estado.geojson, geo/hidro.geojson e geo/subbacia1.geojson
# simplificados e leves, com só as colunas code e name. Depois copie a pasta geo/ para o repositório.
# ==========================================================
# ===== 1) ACESSO =====
!pip -q install pyogrio
from google.colab import drive
drive.mount('/content/drive')

import ee
try:
    ee.Initialize(project="pedrogalve")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="pedrogalve")
print("✅ Drive montado e Earth Engine iniciado")

In [ ]:
# ===== 2) CONFIGURAÇÃO =====
ASSETS = "projects/mapbiomas-agua/assets/territories"
PASTA_SAIDA = "/content/drive/MyDrive/dados/geo"     # depois copie esta pasta para o repositório
PASTA_CSV   = "/content/drive/MyDrive/dados"         # só para conferir se os códigos casam com as planilhas

# simplificação: erro máximo no Earth Engine (m) e tolerância final (graus; 0,01° ≈ 1 km)
NIVEIS = {
    "bioma":     {"asset": f"{ASSETS}/biome",              "csv": "biome-water_surface_monthly_v3.csv",              "erro_m": 1000, "tol": 0.02},
    "estado":    {"asset": f"{ASSETS}/state",              "csv": "state-water_surface_monthly_v3.csv",              "erro_m": 500,  "tol": 0.01},
    "hidro":     {"asset": f"{ASSETS}/hydrographicRegion", "csv": "hydrographicRegion-water_surface_monthly_v3.csv", "erro_m": 1000, "tol": 0.02},
    "subbacia1": {"asset": f"{ASSETS}/subBasinPnrh1",      "csv": "subBasinPnrh1-water_surface_monthly_v3.csv",      "erro_m": 500,  "tol": 0.01},
}
CASAS_DECIMAIS = 4    # 4 casas ≈ 11 m: suficiente para o mapa e deixa o arquivo pequeno
EXCLUIR_NOMES = ["Fernando de Noronha", "Ilha de Fernando de Noronha"]
print("✅ Configuração carregada")

In [ ]:
# ===== 3) FUNÇÕES =====
import os, re, unicodedata
import pandas as pd
import geopandas as gpd
import geemap

CAND_CODE = ["code", "CD_Bioma", "CD_BIOMA", "CD_UF", "RHI_CD", "rhi_cd", "PS1_CD", "PS2_CD", "HYBAS_ID", "CODE", "id", "ID"]
CAND_NOME = ["name", "Bioma", "NM_BIOMA", "NM_UF", "RHI_NM", "rhi_nm", "PS1_NM", "PS2_NM", "NAME", "NOME", "nome"]


def escolher_coluna(colunas, candidatos):
    mapa = {str(c).lower(): c for c in colunas}
    for c in candidatos:
        if c.lower() in mapa:
            return mapa[c.lower()]
    return None


def normalizar_code(x):
    s = re.sub(r"\.0+$", "", str(x).strip())
    return s.lstrip("0") or "0"


def baixar_simplificado(asset, erro_m):
    """Simplifica no próprio Earth Engine (download menor) e traz como GeoDataFrame."""
    fc = ee.FeatureCollection(asset).map(lambda f: ee.Feature(f).simplify(erro_m))
    gdf = geemap.ee_to_gdf(fc)
    return gdf.set_crs("EPSG:4326", allow_override=True)


def preparar(gdf, tol):
    """Só code, name e geometria; simplifica de novo e corrige geometrias inválidas."""
    ccol = escolher_coluna(gdf.columns, CAND_CODE)
    ncol = escolher_coluna(gdf.columns, CAND_NOME)
    if ccol is None or ncol is None:
        raise RuntimeError(f"Não achei colunas de código/nome. Colunas: {list(gdf.columns)}")
    out = gpd.GeoDataFrame({"code": gdf[ccol].map(normalizar_code),
                            "name": gdf[ncol].astype(str).str.strip()},
                           geometry=gdf.geometry, crs="EPSG:4326")
    out = out[~out["name"].isin(EXCLUIR_NOMES)]
    out["geometry"] = out.geometry.simplify(tol, preserve_topology=True).buffer(0)
    out = out[out.geometry.notna() & ~out.geometry.is_empty]
    # um registro por território (algumas camadas vêm em várias partes)
    return out.dissolve(by="code", aggfunc={"name": "first"}).reset_index()


def salvar(gdf, caminho):
    gdf.to_file(caminho, driver="GeoJSON", engine="pyogrio",
                layer_options={"COORDINATE_PRECISION": CASAS_DECIMAIS, "RFC7946": "YES"})
    return os.path.getsize(caminho) / 1e6


def codigos_csv(caminho):
    """Códigos da planilha mensal (para conferir se o mapa vai casar)."""
    with open(caminho, encoding="utf-8", errors="replace") as f:
        cab = f.readline()
    sep = "," if cab.count(",") >= cab.count(";") else ";"
    df = pd.read_csv(caminho, sep=sep, dtype=str)
    ccol = escolher_coluna(df.columns, CAND_CODE + ["codigo"])
    ncol = escolher_coluna(df.columns, CAND_NOME)
    return dict(zip(df[ccol].map(normalizar_code), df[ncol].astype(str).str.strip()))


def chave_nome(s):
    t = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"\s+", " ", t).strip()

print("✅ Funções carregadas")

In [ ]:
# ===== 4) EXPORTAR =====
os.makedirs(PASTA_SAIDA, exist_ok=True)
for ch, nv in NIVEIS.items():
    print(f"⏳ {ch}: baixando {nv['asset']} ...")
    gdf = preparar(baixar_simplificado(nv["asset"], nv["erro_m"]), nv["tol"])
    mb = salvar(gdf, f"{PASTA_SAIDA}/{ch}.geojson")

    # conferência com a planilha mensal: quantos territórios do mapa têm dado
    csv = f"{PASTA_CSV}/{nv['csv']}"
    if os.path.exists(csv):
        cods = codigos_csv(csv)
        nomes = {chave_nome(n) for n in cods.values()}
        ok_cod = gdf["code"].isin(cods.keys()).sum()
        ok_nome = (~gdf["code"].isin(cods.keys()) & gdf["name"].map(chave_nome).isin(nomes)).sum()
        print(f"   ✅ {ch}.geojson: {len(gdf)} territórios, {mb:.2f} MB | casam com a planilha: "
              f"{ok_cod} pelo código + {ok_nome} pelo nome = {ok_cod + ok_nome}/{len(gdf)}")
    else:
        print(f"   ✅ {ch}.geojson: {len(gdf)} territórios, {mb:.2f} MB (planilha não encontrada para conferir)")
    if mb > 3:
        print(f"   ⚠️  arquivo grande: aumente 'tol' de {ch} na célula 2 (ex.: {nv['tol']*2})")

print(f"\n✅ Pronto. Copie a pasta {PASTA_SAIDA} para o repositório, ao lado do index.html (pasta geo/).")